In [1]:
import os
import sys
import re
import scipy as sc
import scipy.stats
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox
import networkx as nx
import shapely
from shapely.ops import split
from tqdm.auto import tqdm
from libpysal.weights import Queen
import pymc as pm
import arviz as az
import pytensor.tensor as pt
from sklearn.cluster import KMeans


sdna_path = os.path.join("C:\\", "Program Files (x86)", "sDNA")
if sdna_path not in sys.path:
    sys.path.append(sdna_path)

import sdnapy
import runcalculation
import sdna_environment

g++ not available, if using conda: `conda install gxx`


In [2]:
# Gemini was used to Refactor the code into its current state

In [3]:
TARGET_CRS = "EPSG:27700"  # British National Grid

# Directory Filepaths
DIR_DATA = os.path.join(os.getcwd(), "economic_data")
DIR_SHP = os.path.join(os.getcwd(), "LSOA_2011_Boundaries_Super_Generalised_Clipped_BSC_EW_V4_-2761400453266118737")
DIR_GRASS_IN = os.path.join(os.getcwd(), "grass_stuff", "inputs")
DIR_GRASS_OUT = os.path.join(os.getcwd(), "grass_stuff", "outputs")
DIR_SDNA_OUT = os.path.join(os.getcwd(), "grass_stuff", "sDNA_outputs")
DIR_ALTERED_IN = os.path.join(os.getcwd(), "grass_stuff", "altered_maps", "input")
DIR_ALTERED_OUT = os.path.join(os.getcwd(), "grass_stuff", "altered_maps", "outputs")

# Create missing directories automatically
for d in [DIR_GRASS_IN, DIR_GRASS_OUT, DIR_SDNA_OUT, DIR_ALTERED_IN, DIR_ALTERED_OUT]:
    os.makedirs(d, exist_ok=True)

# sDNA Analysis Metrics Configuration
SDNA_CONFIG_STR = "radii=500,1000,2000;metric=EUCLIDEAN;nqpdd=1.5;nobetweenness;nohull"
SDNA_MODEL_METRICS = ["NQPDE500",  "NQPDE2000", "NQPDE2000"]
TARGET_COL = "gva_density_2023"

In [4]:
def clean_network_shapefile(input_shapefile: str, output_shapefile: str) -> None:
    """Splits network poly-lines at topological junctions."""
    gdf = gpd.read_file(input_shapefile)
    print("Finding network junctions...")
    merged_lines = shapely.line_merge(shapely.unary_union(gdf.geometry))
    junction_pts_array = shapely.get_parts(shapely.boundary(merged_lines))
    
    print("Building R-Tree spatial index...")
    pts_tree = shapely.STRtree(junction_pts_array)

    print("Splitting lines at junctions...")
    split_geoms = []
    for geom in tqdm(gdf.geometry, desc="Processing roads"):
        local_pt_indices = pts_tree.query(geom, predicate="intersects")
        if len(local_pt_indices) > 0:
            local_junctions = shapely.unary_union(junction_pts_array[local_pt_indices])
            split_geoms.append(split(geom, local_junctions))
        else:
            split_geoms.append(geom)

    split_gdf = gdf.set_geometry(split_geoms).explode(index_parts=False).reset_index(drop=True)
    split_gdf.to_file(output_shapefile)
    print(f"Cleaned network written to: {output_shapefile}")

def clean_duplicate_lines(input_path: str, output_path: str) -> None:
    """Removes bidirectional and duplicate overlapping street segments."""
    gdf = gpd.read_file(input_path)
    normalized_geoms = gdf.geometry.normalize()
    dedup_mask = ~normalized_geoms.to_wkb().duplicated()
    gdf_dedup = gdf[dedup_mask].copy()
    gdf_dedup.to_file(output_path)
    print(f"Deduplicated map written to: {output_path} (Removed {len(gdf) - len(gdf_dedup)} lines)")

def run_sdna_calculation(input_shapefile: str, output_shapefile: str, config_str: str = SDNA_CONFIG_STR) -> None:
    """Executes sDNA Integral calculation across specified networks."""
    env = sdna_environment.SdnaShapefileEnvironment(input_shapefile)
    runcalculation.runcalculation(
        env=env,
        name="sdnaintegral",
        config_string=config_str,
        input_map={"net": input_shapefile},
        output_map={"net": output_shapefile},
    )
    print(f"sDNA process successful. File created: {output_shapefile}")

def mean_aggregation(nqpd_sjoined_df: gpd.GeoDataFrame) -> pd.DataFrame:
    """Aggregates spatial street-segment sDNA metrics up to LSOA geographic boundaries."""
    metrics = [c for c in nqpd_sjoined_df.columns if re.match(r"^NQPDE?\d+$", c, re.IGNORECASE)]
    agg_dict = {col: "mean" for col in metrics}
    return nqpd_sjoined_df.groupby("LSOA11CD").agg(agg_dict).reset_index()

def create_spatial_split(gdf: gpd.GeoDataFrame, test_ratio: float = 0.20, n_clusters: int = 8, seed: int = 42):
    """Splits GeoDataFrame into spatially disjoint train/test clusters using K-Means."""
    coords = np.column_stack([gdf.geometry.centroid.x, gdf.geometry.centroid.y])
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10).fit(coords)
    gdf['spatial_cluster'] = kmeans.labels_
    
    np.random.seed(seed)
    unique_clusters = np.unique(kmeans.labels_)
    test_clusters = np.random.choice(unique_clusters, size=max(1, int(len(unique_clusters) * test_ratio)), replace=False)
    
    gdf['is_test'] = gdf['spatial_cluster'].isin(test_clusters)
    train_gdf = gdf[~gdf['is_test']].copy().reset_index(drop=True)
    test_gdf = gdf[gdf['is_test']].copy().reset_index(drop=True)
    
    return train_gdf, test_gdf

In [5]:
def fit_baseline_model(df_lsoa: gpd.GeoDataFrame, metric_cols: list[str], target_col: str = TARGET_COL):
    
    X = df_lsoa[metric_cols].values
    y = df_lsoa[target_col].values
    
    X_mean, X_std = X.mean(axis=0), X.std(axis=0)
    y_mean, y_std = y.mean(), y.std()
    
    X_scaled = (X - X_mean) / X_std 
    y_scaled = (y - y_mean) / y_std
    
    N, K = X.shape
    
    with pm.Model() as model:
        alpha = pm.Normal("alpha", mu=0, sigma=1)
        beta = pm.Normal("beta", mu=0, sigma=1, shape=K)
        sigma = pm.Exponential("sigma", lam=1)
        nu = pm.Gamma("nu", alpha=2, beta=0.1)
        
        phi_raw = pm.Normal("phi_raw", mu=0, sigma=1, shape=N)
        tau_phi = pm.Exponential("tau_phi", lam=1)
        
        phi = pm.Deterministic("phi", (phi_raw - pt.mean(phi_raw)) / pm.math.sqrt(tau_phi))
        mu = alpha + pm.math.dot(X_scaled, beta) + phi
        
        likelihood = pm.StudentT("y_obs", nu=nu, mu=mu, sigma=sigma, observed=y_scaled)
        
        trace = pm.sample(
            draws=1500,
            tune=1000,
            chains=2,
            target_accept=0.92,
            random_seed=42
        )
        
    scalers = {'X_mean': X_mean, 'X_std': X_std, 'y_mean': y_mean, 'y_std': y_std}
    return model, trace, scalers

def predict_network_delta(
    trace: az.InferenceData, 
    scalers: dict, 
    baseline_df: pd.DataFrame, 
    modified_df: pd.DataFrame, 
    metric_cols: list[str]
) -> pd.DataFrame:
    """Predicts out-of-sample posterior economic shifts (GVA density deltas)."""
    X_base = baseline_df[metric_cols].values
    X_mod = modified_df[metric_cols].values
    
    X_base_scaled = (X_base - scalers['X_mean']) / scalers['X_std']
    X_mod_scaled = (X_mod - scalers['X_mean']) / scalers['X_std']
    
    posterior = trace.posterior
    beta_samples = posterior["beta"].values.reshape(-1, len(metric_cols)).T
    
    delta_X_scaled = X_mod_scaled - X_base_scaled
    delta_mu_scaled = np.dot(delta_X_scaled, beta_samples)
    delta_gva_samples = delta_mu_scaled * scalers['y_std']
    
    results = baseline_df[["LSOA11CD"]].copy()
    results["gva_delta_mean"] = delta_gva_samples.mean(axis=1)
    results["gva_delta_hdi_2.5%"] = np.percentile(delta_gva_samples, 2.5, axis=1)
    results["gva_delta_hdi_97.5%"] = np.percentile(delta_gva_samples, 97.5, axis=1)
    results["prob_positive_growth"] = (delta_gva_samples > 0).mean(axis=1)
    
    return results

In [6]:
#  Downloading Caridff Network data
overpass_date = "2023-12-31T23:59:59Z"
ox.settings.overpass_settings = f'[out:json][timeout:180][date:"{overpass_date}"]'
cardiff_map = ox.graph.graph_from_place("Cardiff, Wales, United Kingdom", network_type="walk", simplify=True)
_, edges = ox.graph_to_gdfs(cardiff_map)
edges = edges.to_crs(TARGET_CRS)

# Clean pandas list columns for shapefile output compatibility
clean_edges = edges.copy()
for col in clean_edges.columns:
    if clean_edges[col].apply(lambda x: isinstance(x, list)).any():
        clean_edges[col] = clean_edges[col].astype(str)

raw_shp_path = os.path.join(DIR_GRASS_IN, "clean_map.shp")
clean_edges.to_file(raw_shp_path)

#  Extracting LSOA Boundaries and Financial GVA Data
lsoa_shape = gpd.read_file(os.path.join(DIR_SHP, "LSOA_2011_EW_BSC_V4.shp"))
lsoa_cardiff = lsoa_shape[lsoa_shape["LSOA11NM"].str.contains("Cardiff")].copy()
lsoa_cardiff = lsoa_cardiff.to_crs(TARGET_CRS)
lsoa_cardiff["area_km2"] = lsoa_cardiff.geometry.area / 1_000_000.0

gva_excel_path = os.path.join(DIR_DATA, "uksmallareagvaestimates1998to2023.xlsx")
df_gva = pd.read_excel(gva_excel_path, sheet_name="Table 2", skiprows=1)
cardiff_gva = df_gva[df_gva["LAD name"] == "Cardiff"].copy()

# Merge GVA Density back to Spatial Polygons
financial_geo = lsoa_cardiff.merge(cardiff_gva, left_on="LSOA11CD", right_on="LSOA code")
financial_geo[TARGET_COL] = financial_geo["2023"] / financial_geo["area_km2"]

In [7]:
# Clean and run sDNA on Baseline Network
base_cleaned_shp = os.path.join(DIR_GRASS_OUT, "clean_map_cleaned.shp")
base_sdna_ready = os.path.join(DIR_GRASS_OUT, "clean_map_sdna_ready.shp")
base_sdna_output = os.path.join(DIR_SDNA_OUT, "cardiff_nqpd_sdna.shp")

clean_network_shapefile(raw_shp_path, base_cleaned_shp)
clean_duplicate_lines(base_cleaned_shp, base_sdna_ready)
run_sdna_calculation(base_sdna_ready, base_sdna_output)

# Join Segment Results via Centroid to eliminate road boundary overlaps
nqpd_results = gpd.read_file(base_sdna_output)
nqpd_centroids = nqpd_results.copy()
nqpd_centroids["geometry"] = nqpd_centroids.geometry.centroid

nqpd_per_lsoa = gpd.sjoin(
    nqpd_centroids, 
    lsoa_cardiff[["LSOA11CD", "geometry"]], 
    how="inner", 
    predicate="within"
)

# Aggregate street level data to LSOA Polygons
mean_agg_df = mean_aggregation(nqpd_per_lsoa)
# final_bayes_df = mean_agg_df.merge(financial_geo, left_on="LSOA11CD", right_on="LSOA code", how="left")
final_bayes_df = financial_geo.merge(mean_agg_df, left_on="LSOA code", right_on="LSOA11CD", how="left")

Finding network junctions...
Building R-Tree spatial index...
Splitting lines at junctions...


Processing roads:   0%|          | 0/87434 [00:00<?, ?it/s]

Cleaned network written to: c:\Users\stefa\OneDrive\Masters\Dissertation\Maps2\grass_stuff\outputs\clean_map_cleaned.shp
Deduplicated map written to: c:\Users\stefa\OneDrive\Masters\Dissertation\Maps2\grass_stuff\outputs\clean_map_sdna_ready.shp (Removed 43729 lines)
sDNA sdnaintegral config: radii=500,1000,2000;metric=EUCLIDEAN;nqpdd=1.5;nobetweenness;nohull
Loading DLL C:\Program Files (x86)\sDNA\\x64\\sdna_vs2008.dll
  Using default of euclidean link centres for euclidean analysis
  Using xytolerance=0, ztolerance=0
Input has 43705 rows
Creating geometry output c:\Users\stefa\OneDrive\Masters\Dissertation\Maps2\grass_stuff\sDNA_outputs\cardiff_nqpd_sdna.shp
Adding fields:
    ID         - ID
    LConn      - Line Connectivity
    LLen       - Line Length
    LFrac      - Link Fraction
    LAC        - Line Ang Curvature
    LSin       - Line Sinuosity
    LBear      - Line Bearing
    MED500     - Mean Euc Dist R500
    NQPDE500   - NetQuantPD Euc R500
    Lnk500     - Links R500
  

Progress: 100%
Progress: 0.0%

sDNA processing
  sDNA is running in 64-bit mode
  Building network and checking for tolerance errors...
  Running sDNA Integral calculation...


Progress: 100%%
Progress: 0.0%

TIME 2625.3 for radii=500,1000,2000;metric=EUCLIDEAN;nqpdd=1.5;nobetweenness;nohull
Writing c:\Users\stefa\OneDrive\Masters\Dissertation\Maps2\grass_stuff\sDNA_outputs\cardiff_nqpd_sdna.shp


Progress: 100%


sDNA process successful. File created: c:\Users\stefa\OneDrive\Masters\Dissertation\Maps2\grass_stuff\sDNA_outputs\cardiff_nqpd_sdna.shp


In [8]:
final_bayes_df = financial_geo.merge(mean_agg_df, left_on="LSOA code", right_on="LSOA11CD", how="left")

In [10]:
# Paths for counterfactual map variations
altered_path = os.path.join(DIR_ALTERED_IN, "altered_map.shp")
altered_cleaned = os.path.join(DIR_ALTERED_OUT, "altered_map_cleaned.shp")
altered_sdna_ready = os.path.join(DIR_ALTERED_OUT, "altered_map_sdna_ready.shp")
altered_sdna_output = os.path.join(DIR_SDNA_OUT, "altered_nqpd_sdna.shp")

# Clean and Run sDNA on Proposed Network Layout
clean_network_shapefile(altered_path, altered_cleaned)
clean_duplicate_lines(altered_cleaned, altered_sdna_ready)
run_sdna_calculation(altered_sdna_ready, altered_sdna_output)

# Join and aggregate altered road metrics to LSOAs
modified_nqpd = gpd.read_file(altered_sdna_output)
modified_centroids = modified_nqpd.copy()
modified_centroids["geometry"] = modified_centroids.geometry.centroid

modified_per_lsoa = gpd.sjoin(
    modified_centroids, 
    lsoa_cardiff[["LSOA11CD", "geometry"]], 
    how="inner", 
    predicate="within"
)
modified_mean_agg = mean_aggregation(modified_per_lsoa)

Finding network junctions...
Building R-Tree spatial index...
Splitting lines at junctions...


Processing roads:   0%|          | 0/110608 [00:00<?, ?it/s]

Cleaned network written to: c:\Users\stefa\OneDrive\Masters\Dissertation\Maps2\grass_stuff\altered_maps\outputs\altered_map_cleaned.shp
Deduplicated map written to: c:\Users\stefa\OneDrive\Masters\Dissertation\Maps2\grass_stuff\altered_maps\outputs\altered_map_sdna_ready.shp (Removed 55304 lines)
sDNA sdnaintegral config: radii=500,1000,2000;metric=EUCLIDEAN;nqpdd=1.5;nobetweenness;nohull
  Using default of euclidean link centres for euclidean analysis
  Using xytolerance=0, ztolerance=0
Input has 55304 rows
Creating geometry output c:\Users\stefa\OneDrive\Masters\Dissertation\Maps2\grass_stuff\sDNA_outputs\altered_nqpd_sdna.shp
Adding fields:
    ID         - ID
    LConn      - Line Connectivity
    LLen       - Line Length
    LFrac      - Link Fraction
    LAC        - Line Ang Curvature
    LSin       - Line Sinuosity
    LBear      - Line Bearing
    MED500     - Mean Euc Dist R500
    NQPDE500   - NetQuantPD Euc R500
    Lnk500     - Links R500
    Len500     - Length R500
    A

Progress: 100%
Progress: 0.0%

sDNA processing
  sDNA is running in 64-bit mode
  Building network and checking for tolerance errors...
  Running sDNA Integral calculation...


Progress: 100%%
Progress: 0.0%

TIME 4166.2 for radii=500,1000,2000;metric=EUCLIDEAN;nqpdd=1.5;nobetweenness;nohull
Writing c:\Users\stefa\OneDrive\Masters\Dissertation\Maps2\grass_stuff\sDNA_outputs\altered_nqpd_sdna.shp


Progress: 100%


sDNA process successful. File created: c:\Users\stefa\OneDrive\Masters\Dissertation\Maps2\grass_stuff\sDNA_outputs\altered_nqpd_sdna.shp


In [14]:
# Second iteration of model, (Gemini generated, to help implement ICAR)

def fit_baseline_model(df_lsoa, metric_cols: list[str], target_col: str = "gva_density_2023"):
    """
    Fits Bayesian Spatial Regression with ICAR Random Effects and Student-t Likelihood.
    Uses sparse adjacency indexing to prevent PyMC matrix shape mismatches.
    """
    # 1. Prepare Adjacency Data for ICAR
    w = Queen.from_dataframe(df_lsoa, use_index=False)
    
    # Extract neighbor counts (node degrees) and sparse edge pairs
    node_degrees = np.array([len(neighbors) for neighbors in w.neighbors.values()])
    
    node1 = []
    node2 = []
    for node, neighbors in w.neighbors.items():
        for neighbor in neighbors:
            if node < neighbor:  # Keep unique undirected edges
                node1.append(node)
                node2.append(neighbor)
                
    node1 = np.array(node1, dtype=int)
    node2 = np.array(node2, dtype=int)
    
    # Feature & Target Extraction
    X = df_lsoa[metric_cols].values
    y = df_lsoa[target_col].values
    
    X_mean, X_std = X.mean(axis=0), X.std(axis=0)
    y_mean, y_std = y.mean(), y.std()
    
    # Safe scaling against zero-variance
    X_std[X_std == 0] = 1.0
    
    X_scaled = (X - X_mean) / X_std 
    y_scaled = (y - y_mean) / y_std
    
    N, K = X.shape
    
    with pm.Model() as model:
        # Global Intercept & Standardized Beta Priors
        alpha = pm.Normal("alpha", mu=0, sigma=0.5)
        beta = pm.Normal("beta", mu=0, sigma=0.5, shape=K)
        
        # Likelihood Dispersion Parameters
        sigma = pm.Exponential("sigma", lam=2)
        nu = pm.Gamma("nu", alpha=3, beta=0.2)
        
        # Spatial Precision Scale
        sigma_phi = pm.Exponential("sigma_phi", lam=1)
        
        # Native ICAR Random Effect for Spatial Smoothness
        phi = pm.ICAR("phi", W=w.full()[0], sigma=sigma_phi)
        
        # Deterministic Equation
        mu = alpha + pm.math.dot(X_scaled, beta) + phi
        
        # Student-t Likelihood
        likelihood = pm.StudentT("y_obs", nu=nu, mu=mu, sigma=sigma, observed=y_scaled)
        
        # Sampler Settings Optimized for Spatial Geometry
        trace = pm.sample(
            draws=1500,
            tune=1000,
            chains=4,
            target_accept=0.95,
            max_treedepth=12,
            random_seed=42,
            init="adapt_diag"
        )
        
    scalers = {'X_mean': X_mean, 'X_std': X_std, 'y_mean': y_mean, 'y_std': y_std}
    return model, trace, scalers

In [16]:
train_df, test_df = create_spatial_split(final_bayes_df, test_ratio=0.20)

print("--- STAGE 1: Validating Model Out-Of-Sample Generalization ---")
val_model, val_trace, val_scalers = fit_baseline_model(train_df, SDNA_MODEL_METRICS)

# Generalization Metrics Evaluation
X_test_scaled = (test_df[SDNA_MODEL_METRICS].values - val_scalers['X_mean']) / val_scalers['X_std']
beta_samples = val_trace.posterior["beta"].values.reshape(-1, len(SDNA_MODEL_METRICS)).T
alpha_samples = val_trace.posterior["alpha"].values.flatten()

mu_test_samples = (alpha_samples + np.dot(X_test_scaled, beta_samples)) * val_scalers['y_std'] + val_scalers['y_mean']
y_test_true = test_df[TARGET_COL].values
y_test_pred = mu_test_samples.mean(axis=1)

print(f"Validation Test MAE:  £{np.mean(np.abs(y_test_true - y_test_pred)):.2f}/km²")
print(f"Validation Test RMSE: £{np.sqrt(np.mean((y_test_true - y_test_pred)**2)):.2f}/km²")

print("\n--- STAGE 2: Fitting Production Model on Full Master Dataset ---")
final_model, final_trace, final_scalers = fit_baseline_model(final_bayes_df, SDNA_MODEL_METRICS)

Initializing NUTS using adapt_diag...


--- STAGE 1: Validating Model Out-Of-Sample Generalization ---


Multiprocess sampling (4 chains in 4 jobs)
NUTS: [alpha, beta, sigma, nu, sigma_phi, phi]


Output()

Sampling 4 chains for 1_000 tune and 1_500 draw iterations (4_000 + 6_000 draws total) took 380 seconds.
Chain 2 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Validation Test MAE:  £223.73/km²
Validation Test RMSE: £439.82/km²

--- STAGE 2: Fitting Production Model on Full Master Dataset ---


Initializing NUTS using adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [alpha, beta, sigma, nu, sigma_phi, phi]


Output()

Sampling 4 chains for 1_000 tune and 1_500 draw iterations (4_000 + 6_000 draws total) took 256 seconds.
There were 857 divergences after tuning. Increase `target_accept` or reparameterize.
Chain 3 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


In [9]:
# Perform split of training and testing data
train_df, test_df = create_spatial_split(final_bayes_df, test_ratio=0.20)

print("--- STAGE 1: Validating Model Out-Of-Sample Generalization ---")
val_model, val_trace, val_scalers = fit_baseline_model(train_df, SDNA_MODEL_METRICS)

# Generalization Metrics Evaluation
X_test_scaled = (test_df[SDNA_MODEL_METRICS].values - val_scalers['X_mean']) / val_scalers['X_std']
beta_samples = val_trace.posterior["beta"].values.reshape(-1, len(SDNA_MODEL_METRICS)).T
alpha_samples = val_trace.posterior["alpha"].values.flatten()

mu_test_samples = (alpha_samples + np.dot(X_test_scaled, beta_samples)) * val_scalers['y_std'] + val_scalers['y_mean']
y_test_true = test_df[TARGET_COL].values
y_test_pred = mu_test_samples.mean(axis=1)

print(f"Validation Test MAE:  £{np.mean(np.abs(y_test_true - y_test_pred)):.2f}/km²")
print(f"Validation Test RMSE: £{np.sqrt(np.mean((y_test_true - y_test_pred)**2)):.2f}/km²")

print("\n--- STAGE 2: Fitting Production Model on Full Master Dataset ---")
final_model, final_trace, final_scalers = fit_baseline_model(final_bayes_df, SDNA_MODEL_METRICS)

--- STAGE 1: Validating Model Out-Of-Sample Generalization ---


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha, beta, sigma, nu, phi_raw, tau_phi]


Output()

Sampling 2 chains for 1_000 tune and 1_500 draw iterations (2_000 + 3_000 draws total) took 184 seconds.
There were 7 divergences after tuning. Increase `target_accept` or reparameterize.
Chain 0 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Chain 1 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Validation Test MAE:  £131.60/km²
Validation Test RMSE: £425.73/km²

--- STAGE 2: Fitting Production Model on Full Master Dataset ---


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha, beta, sigma, nu, phi_raw, tau_phi]


Output()

Sampling 2 chains for 1_000 tune and 1_500 draw iterations (2_000 + 3_000 draws total) took 142 seconds.
There were 27 divergences after tuning. Increase `target_accept` or reparameterize.
Chain 0 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Chain 1 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


In [11]:
# Force-assign LSOA11CD back onto financial_geo prior to merging
financial_geo["LSOA11CD"] = financial_geo["LSOA code"]

# Construct final_bayes_df retaining LSOA11CD
final_bayes_df = financial_geo.merge(
    mean_agg_df, 
    on="LSOA11CD", 
    how="left"
)

In [12]:
# 3. Forecast Bayesian Deliberative Impacts
economic_forecast = predict_network_delta(
    trace=final_trace,
    scalers=final_scalers,
    baseline_df=final_bayes_df,
    modified_df=modified_mean_agg,
    metric_cols=SDNA_MODEL_METRICS
)

# 4. Save Final Shapefile Output for Visualization
forecast_gdf = lsoa_cardiff.merge(economic_forecast, on="LSOA11CD")
output_gpkg = os.path.join(os.getcwd(), "cardiff_gva_forecast_results.gpkg")
forecast_gdf.to_file(output_gpkg, driver="GPKG")
print(f"Pipeline complete! Policy spatial forecasts exported to: {output_gpkg}")

Pipeline complete! Policy spatial forecasts exported to: c:\Users\stefa\OneDrive\Masters\Dissertation\Maps2\cardiff_gva_forecast_results.gpkg


In [13]:
forecast_gdf

,LSOA11CD,LSOA11NM,LSOA11NMW,BNG_E,BNG_N,LONG,LAT,GlobalID,geometry,area_km2,gva_delta_mean,gva_delta_hdi_2.5%,gva_delta_hdi_97.5%,prob_positive_growth
0,W01001694,Cardiff 036A,Caerdydd 036A,319725,176939,-3.15751,51.4857,8f0388a2-395c-4dca-90fe-a39f9a4a91e0,"POLYGON ((320102.857 176993.3, 319985.048 1768...",0.175056,28.045611,26.101327,29.991709,1.0
1,W01001695,Cardiff 036B,Caerdydd 036B,319089,176474,-3.16656,51.4814,d86d2e79-f363-4ed0-a78d-a8fd7dd84b94,"POLYGON ((319411.518 176308.72, 318743.68 1760...",0.369357,31.668314,29.521537,33.819120,1.0
2,W01001696,Cardiff 036C,Caerdydd 036C,319609,176661,-3.15911,51.4832,dd605a50-653c-455e-80e3-1be2583e6623,"POLYGON ((319954.249 176790.215, 319863.29 176...",0.180205,23.745359,22.074866,25.417100,1.0
3,W01001697,Cardiff 036D,Caerdydd 036D,320349,177458,-3.14864,51.4904,2455d541-ff6a-435c-8b5b-b6568b6b0a26,"POLYGON ((320769.41 177854.193, 320102.857 176...",0.183144,28.649032,25.248371,31.999770,1.0
4,W01001698,Cardiff 036E,Caerdydd 036E,319905,177301,-3.15500,51.4890,c8e48c66-86b3-4e6a-940e-965906a6754e,"POLYGON ((320213.023 177544.612, 320196.554 17...",0.166348,27.438206,26.062843,28.824443,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209,W01001950,Cardiff 049D,Caerdydd 049D,318666,174758,-3.17225,51.4659,20850304-390c-446f-8964-02f64f92da55,"POLYGON ((318781.568 174738.145, 318632.45 174...",0.075724,-21.782779,-22.930007,-20.646955,0.0
210,W01001951,Cardiff 049E,Caerdydd 049E,319130,175914,-3.16584,51.4764,b46bbe45-852c-4c33-bb4e-f8fde30c6626,"POLYGON ((319551.524 175706.412, 319349.721 17...",0.406890,-4.015176,-4.201438,-3.829545,0.0
211,W01001952,Cardiff 049F,Caerdydd 049F,318405,175403,-3.17616,51.4717,39b58630-f0a0-4f78-9507-9c773a8e3b8f,"POLYGON ((318746.434 175688.431, 318472.001 17...",0.585147,7.940392,7.422642,8.461227,1.0
212,W01001953,Cardiff 005G,Caerdydd 005G,316859,181748,-3.19991,51.5285,23d199a7-e631-4761-a779-3cbb346f6f31,"POLYGON ((317224.001 181570, 317222.001 181122...",0.432330,38.226402,36.050442,40.432836,1.0
